In [1]:
import torch
import importlib.util

print("PyTorch:", torch.__version__)
print("Transformers installed:", importlib.util.find_spec("transformers") is not None)
print("Datasets installed:", importlib.util.find_spec("datasets") is not None)
print("GPU available:", torch.cuda.is_available())
print("TPU environment:", importlib.util.find_spec("torch_xla") is not None)

PyTorch: 2.11.0+cu128
Transformers installed: True
Datasets installed: True
GPU available: True
TPU environment: False


In [2]:
# الخلية 1: استيراد المكتبات وتحميل البيانات وفحصها

from datasets import load_dataset

# تحميل مجموعة البيانات العربية التي تحتوي على معلومات شخصية
dataset = load_dataset("Bisher/synthetic-arabic-pii")

# عرض الأقسام والأعمدة الموجودة
print("أقسام البيانات:", list(dataset.keys()))

for split in dataset:
    print(f"{split}: {len(dataset[split])} صف")

# عرض مثال للتأكد من شكل البيانات والتصنيفات
sample = dataset["train"][0]

print("\nالأعمدة:", list(sample.keys()))
print("\nالنص:")
print(sample["text"])

print("\nالمعلومات الشخصية وتصنيفاتها:")
print(sample["entities"])

README.md:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.52MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split:   0%|          | 0/11979 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

أقسام البيانات: ['train', 'validation', 'test']
train: 11979 صف
validation: 2000 صف
test: 3000 صف

الأعمدة: ['id', 'text', 'entities', 'split', 'source', 'row_type', 'metadata']

النص:
**العميل:** السلام عليكم، أود الاستفسار عن سبب انقطاع الخدمة عن خط الهاتف المحمول الخاص بي، فقد توقفت الخدمة فجأة منذ يومين.

**الدعم:** وعليكم السلام، يسعدني مساعدتك. أرجو تزويدي برقم خطك أو رقم هاتفك للتحقق من الحساب.

**العميل:** رقم الهاتف هو +212 706 898 160، والخط مسجل باسمي أنا عبد الولي العباس.

**الدعم:** شكراً لك، سأتحقق من الحساب فوراً. يبدو أن هناك مشكلة في الفاتورة السابقة، حيث لم يتم سدادها في الموعد المحدد. هل ترغب في تفعيل الخدمة بعد تسوية المبلغ؟

**العميل:** نعم، من فضلك أبلغني بالمبلغ المطلوب وكيفية السداد، لأنني بحاجة ماسة للخط في عملي.

المعلومات الشخصية وتصنيفاتها:
[{'start': 247, 'end': 263, 'label': 'PHONE_NUMBER', 'text': '+212 706 898 160'}, {'start': 286, 'end': 302, 'label': 'PERSON', 'text': 'عبد الولي العباس'}]


In [3]:
# الخلية 2: تحميل نموذج سِتر على معالج GPU

from transformers import pipeline

MODEL = "mabahboh/sitr-privacy-filter-ar-v1"

ner = pipeline(
    "token-classification",
    model=MODEL,
    aggregation_strategy="simple",
    device=0
)

print("تم تحميل نموذج سِتر بنجاح")

config.json:   0%|          | 0.00/6.03k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.80GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 27.9MB            

tokenizer.json: downloading bytes:           |  0.00B            

تم تحميل نموذج سِتر بنجاح


In [4]:
# الخلية 3: اختبار نموذج سِتر على نص يحتوي على معلومات شخصية

text = "اسمي خالد العتيبي ورقم هاتفي 0501234567 ورقم هويتي 1023456789"

# اكتشاف المعلومات الشخصية في النص
results = ner(text)

# عرض المعلومات التي اكتشفها النموذج
for item in results:
    print(item["entity_group"], "|", item["word"], "|", round(item["score"], 2))

private_phone | 050123456 | 0.53
national_id | 7 | 0.48
national_id | 1023456789 | 0.94


In [5]:
# الخلية 4: فحص تصنيفات المعلومات الشخصية في بيانات الاختبار

test_data = dataset["test"]

print("عدد أمثلة الاختبار:", len(test_data))
print("\nمثال على التصنيفات الصحيحة:")

for entity in test_data[0]["entities"]:
    print(entity)

عدد أمثلة الاختبار: 3000

مثال على التصنيفات الصحيحة:
{'start': 6, 'end': 15, 'label': 'PERSON', 'text': 'سمر الفقي'}
{'start': 50, 'end': 73, 'label': 'BANK_ACCOUNT_NUMBER', 'text': 'حساب-EG-266-71713698122'}


In [6]:
# الخلية 5: تقييم نموذج سِتر مع توحيد التصنيفات والسماح بتداخل النصوص

from collections import Counter

# توحيد أسماء الفئات بين النموذج والبيانات
def category(label):
    label = label.lower()

    if "phone" in label:
        return "PHONE"
    if "person" in label or "name" in label:
        return "PERSON"
    if "email" in label:
        return "EMAIL"
    if "iban" in label:
        return "IBAN"
    if "bank" in label or "account" in label:
        return "BANK_ACCOUNT"
    if "national_id" in label or "identity" in label:
        return "ID"
    if "address" in label or "street" in label or "city" in label:
        return "ADDRESS"

    return None

scores = Counter()

# تجربة أول 100 نص من بيانات الاختبار
tested = dataset["test"].select(range(100))

for row in tested:
    true_entities = [
        (e["start"], e["end"], category(e["label"]))
        for e in row["entities"]
        if category(e["label"]) is not None
    ]

    predicted_entities = [
        (e["start"], e["end"], category(e["entity_group"]))
        for e in ner(row["text"])
        if category(e["entity_group"]) is not None
    ]

    # مطابقة كل اكتشاف مع معلومة صحيحة واحدة فقط
    matched = set()

    for start, end, label in predicted_entities:
        found = False

        for i, (true_start, true_end, true_label) in enumerate(true_entities):
            if i not in matched and label == true_label:
                if start < true_end and end > true_start:
                    matched.add(i)
                    found = True
                    break

        scores["TP" if found else "FP"] += 1

    scores["FN"] += len(true_entities) - len(matched)

tp, fp, fn = scores["TP"], scores["FP"], scores["FN"]

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print("Texts tested:", len(tested))
print("TP:", tp, "| FP:", fp, "| FN:", fn)
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1:", round(f1, 3))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Texts tested: 100
TP: 91 | FP: 151 | FN: 68
Precision: 0.376
Recall: 0.572
F1: 0.454


In [8]:
# الخلية 6: إخفاء المعلومات الشخصية من نص مكالمة
import re
# نص مكالمة تجريبية بمعلومات شخصية اصطناعية
text = """
العميل: السلام عليكم، رقم جوالي 0501234567
ورقم هويتي 1023456789 وإيميلي test@example.com.
الدعم: شكرًا لك، سأتحقق من بياناتك.
"""
# جمع المعلومات التي يكتشفها النموذج
spans = [(e["start"], e["end"]) for e in ner(text)]
# إضافة المعلومات التي يكتشفها Regex
patterns = [
    r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}",  # الإيميل
    r"(?<!\d)05\d{8}(?!\d)",             # الجوال السعودي
    r"\b1\d{9}\b|\b2\d{9}\b",            # رقم الهوية
    r"\bSA\d{22}\b"                      # الآيبان السعودي
]
for pattern in patterns:
    spans.extend(
        (m.start(), m.end())
        for m in re.finditer(pattern, text, re.IGNORECASE)
    )
# دمج المواقع المتداخلة حتى لا يبقى جزء من المعلومة ظاهرًا
spans.sort()
merged = []
for start, end in spans:
    if merged and start < merged[-1][1]:
        merged[-1] = (merged[-1][0], max(merged[-1][1], end))
    else:
        merged.append((start, end))
# استبدال المعلومات من نهاية النص إلى بدايته
redacted = text
for start, end in reversed(merged):
    redacted = redacted[:start] + "[REDACTED]" + redacted[end:]
print("النص الأصلي:\n", text)
print("\nالنص بعد الإخفاء:\n", redacted)

النص الأصلي:
 
العميل: السلام عليكم، رقم جوالي 0501234567
ورقم هويتي 1023456789 وإيميلي test@example.com.
الدعم: شكرًا لك، سأتحقق من بياناتك.


النص بعد الإخفاء:
 
العميل: السلام عليكم، رقم جوالي [REDACTED]
ورقم هويتي [REDACTED] وإيميلي [REDACTED].
الدعم: شكرًا لك، سأتحقق من بياناتك.



In [11]:
%pip install -q openai-whisper

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [12]:
# الخلية 7: تحويل تسجيل المكالمة إلى نص باستخدام Whisper

import whisper

# مسار التسجيل الصوتي
audio_path = "/kaggle/input/datasets/reenamohammed/callsense-call051/Call_051.m4a"

# تحميل نموذج Whisper الصغير
whisper_model = whisper.load_model("small")

# تحويل التسجيل إلى نص باللغة العربية
result = whisper_model.transcribe(
    audio_path,
    language="ar"
)

# عرض النص الناتج
call_transcript = result["text"]

print("نص المكالمة:")
print(call_transcript)

100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 93.9MiB/s]


نص المكالمة:
 مرحبا بكم في خدمة العملاء كيف يمكنني مساعدتكم؟ مرحبا بكم، التصدف خصوص فاتورة هذا الشهر فقد خصم المبلغ مراتين من حساب المصرفين اعتدت عن هذا الازعاج، هل يمكنكم تزويد يبرقم الجوال المرطمز بالحساب؟ الرقم 05 5 0 1 2 3 4 5 وهذا الثالث مرة التصدف فيها بسبب المشكلة نفسها وفي كل مرة يقال لي أنها ستحال خلال اليومين ولم تحال ارى أن الطلب السابق ليزال قيد المراجعة قيد المراجعة من 3 اسابيه هذا غير مقبول اطلاقا ولم يعب لديه اي ثقى بيوعدكم اتفهم انزعاجكم تماما وسوى اتابع الطلب بنفسي وأوفيكم بالنتيجة خلال 48 ساعة لا اريد وعودة جديدة اريد استرداد مبلغ اليوم واتلب التحدث مع المشرف حالا حسنا سحاول المكالمة المشرف فورا ولا نغلق الخطق قبل ان يثلمها شكرا لك وان لم تحل المسأل اليوم فسأقدم شكرا رسمي


In [13]:
# الخلية 8: إخفاء المعلومات الشخصية من المكالمة

import re

text = call_transcript

# اكتشاف المعلومات باستخدام Sitr
entities = ner(text)
spans = [(e["start"], e["end"]) for e in entities]

# اكتشاف أرقام الجوال حتى لو كانت الأرقام مفصولة بمسافات
for match in re.finditer(r"(?<!\d)0\s*5(?:\s*\d){8}(?!\d)", text):
    spans.append((match.start(), match.end()))

# دمج المواقع المتداخلة
spans.sort()
merged = []

for start, end in spans:
    if merged and start < merged[-1][1]:
        merged[-1] = (merged[-1][0], max(merged[-1][1], end))
    else:
        merged.append((start, end))

# إخفاء المعلومات من نهاية النص إلى بدايته
redacted_transcript = text

for start, end in reversed(merged):
    redacted_transcript = (
        redacted_transcript[:start]
        + "[REDACTED]"
        + redacted_transcript[end:]
    )

print("النص بعد الإخفاء:")
print(redacted_transcript)

النص بعد الإخفاء:
 مرحبا بكم في خدمة العملاء كيف يمكنني مساعدتكم؟ مرحبا بكم، التصدف خصوص فاتورة هذا الشهر فقد خصم المبلغ مراتين من حساب المصرفين اعتدت عن هذا الازعاج، هل يمكنكم تزويد يبرقم الجوال المرطمز بالحساب؟ الرقم 05 5 0[REDACTED]1[REDACTED]3 4 5 وهذا الثالث مرة التصدف فيها بسبب المشكلة نفسها وفي كل مرة يقال لي أنها ستحال خلال اليومين ولم تحال ارى أن الطلب السابق ليزال قيد المراجعة قيد المراجعة من 3 اسابيه هذا غير مقبول اطلاقا ولم يعب لديه اي ثقى بيوعدكم اتفهم انزعاجكم تماما وسوى اتابع الطلب بنفسي وأوفيكم بالنتيجة خلال 48 ساعة لا اريد وعودة جديدة اريد استرداد مبلغ اليوم واتلب التحدث مع المشرف حالا حسنا سحاول المكالمة المشرف فورا ولا نغلق الخطق قبل ان يثلمها شكرا لك وان لم تحل المسأل اليوم فسأقدم شكرا رسمي


In [14]:
# الخلية 8: إخفاء المعلومات الشخصية من المكالمة

import re

text = call_transcript

# اكتشاف المعلومات باستخدام Sitr
entities = ner(text)
spans = [(e["start"], e["end"]) for e in entities]

# اكتشاف تسلسل أرقام الجوال حتى لو فصل Whisper الأرقام بمسافات
pattern = r"(?<!\d)0(?:\s*\d){8,11}(?!\d)"

for match in re.finditer(pattern, text):
    spans.append((match.start(), match.end()))

# دمج المواقع المتداخلة
spans.sort()
merged = []

for start, end in spans:
    if merged and start <= merged[-1][1]:
        merged[-1] = (
            merged[-1][0],
            max(merged[-1][1], end)
        )
    else:
        merged.append((start, end))

# إخفاء المعلومات من نهاية النص إلى بدايته
redacted_transcript = text

for start, end in reversed(merged):
    redacted_transcript = (
        redacted_transcript[:start]
        + "[REDACTED]"
        + redacted_transcript[end:]
    )

print("النص بعد الإخفاء:")
print(redacted_transcript)

النص بعد الإخفاء:
 مرحبا بكم في خدمة العملاء كيف يمكنني مساعدتكم؟ مرحبا بكم، التصدف خصوص فاتورة هذا الشهر فقد خصم المبلغ مراتين من حساب المصرفين اعتدت عن هذا الازعاج، هل يمكنكم تزويد يبرقم الجوال المرطمز بالحساب؟ الرقم [REDACTED] وهذا الثالث مرة التصدف فيها بسبب المشكلة نفسها وفي كل مرة يقال لي أنها ستحال خلال اليومين ولم تحال ارى أن الطلب السابق ليزال قيد المراجعة قيد المراجعة من 3 اسابيه هذا غير مقبول اطلاقا ولم يعب لديه اي ثقى بيوعدكم اتفهم انزعاجكم تماما وسوى اتابع الطلب بنفسي وأوفيكم بالنتيجة خلال 48 ساعة لا اريد وعودة جديدة اريد استرداد مبلغ اليوم واتلب التحدث مع المشرف حالا حسنا سحاول المكالمة المشرف فورا ولا نغلق الخطق قبل ان يثلمها شكرا لك وان لم تحل المسأل اليوم فسأقدم شكرا رسمي


In [15]:
# الخلية 9: اختبار اكتشاف أسماء الأشخاص

from transformers import pipeline

# تحميل نموذج عربي للتعرف على الكيانات
name_model = pipeline(
    "token-classification",
    model="hatmimoha/arabic-ner",
    aggregation_strategy="simple",
    device=0
)

# تجربة النموذج على مثال عربي يحتوي على اسم
sample_text = "اسمي خالد العتيبي وأرغب في الاستفسار عن فاتورتي."

# اكتشاف الكيانات في النص
results = name_model(sample_text)

# عرض الأسماء المكتشفة
print("النص:", sample_text)
print("\nالكيانات المكتشفة:")

for item in results:
    print(item["entity_group"], "|", item["word"], "|", round(item["score"], 2))

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/334k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/837k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

النص: اسمي خالد العتيبي وأرغب في الاستفسار عن فاتورتي.

الكيانات المكتشفة:
PERSON | خالد العتيبي | 1.0


In [16]:
# الخلية 10: اختبار اكتشاف الأسماء على عينة من البيانات
name_test_texts = dataset["test"].select(range(100))["text"]

total_names = 0
detected_names = 0

for text_sample in name_test_texts:
    actual_entities = ner(text_sample)  # نموذج سِتر
    predicted_names = name_model(text_sample)

    actual_names = [
        item for item in actual_entities
        if "person" in item["entity_group"].lower()
        or "name" in item["entity_group"].lower()
    ]

    detected_names += len(predicted_names)
    total_names += len(actual_names)

print("عدد الأسماء التي اكتشفها نموذج سِتر:", total_names)
print("عدد الكيانات التي اكتشفها نموذج الأسماء:", detected_names)

عدد الأسماء التي اكتشفها نموذج سِتر: 0
عدد الكيانات التي اكتشفها نموذج الأسماء: 325


In [18]:
# فحص تصنيفات الكيانات في النموذجين
sample_text = dataset["test"][0]["text"]

print("النص:")
print(sample_text)

print("\nمخرجات سِتر:")
for item in ner(sample_text):
    print(item["entity_group"], "|", item["word"])

print("\nمخرجات نموذج الأسماء:")
for item in name_model(sample_text):
    print(item["entity_group"], "|", item["word"])

النص:
بص يا سمر الفقي، أنا حولتلك الفلوس على الحساب ده: حساب-EG-266-71713698122.

مخرجات سِتر:
api_key | G
passport_number | -
gulf_iban | 266
account_number | -
gulf_iban | 71713698122

مخرجات نموذج الأسماء:
PERSON | سمر الفقي


In [19]:
# فحص طريقة تسجيل الكيانات في البيانات
sample = dataset["test"][0]

print("النص:")
print(sample["text"])

print("\nالكيانات المسجلة:")
print(sample["entities"])

النص:
بص يا سمر الفقي، أنا حولتلك الفلوس على الحساب ده: حساب-EG-266-71713698122.

الكيانات المسجلة:
[{'start': 6, 'end': 15, 'label': 'PERSON', 'text': 'سمر الفقي'}, {'start': 50, 'end': 73, 'label': 'BANK_ACCOUNT_NUMBER', 'text': 'حساب-EG-266-71713698122'}]


In [20]:
# الخلية 11: تقييم نموذج اكتشاف الأسماء
from collections import Counter

TP = 0
FP = 0
FN = 0

for sample in dataset["test"].select(range(100)):
    text = sample["text"]

    actual_names = {
        entity["text"]
        for entity in sample["entities"]
        if entity["label"] == "PERSON"
    }

    predicted_names = {
        item["word"].replace("##", "").strip()
        for item in name_model(text)
        if item["entity_group"] == "PERSON"
    }

    for name in predicted_names:
        if name in actual_names:
            TP += 1
        else:
            FP += 1

    for name in actual_names:
        if name not in predicted_names:
            FN += 1

precision = TP / (TP + FP) if TP + FP else 0
recall = TP / (TP + FN) if TP + FN else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("Samples tested:", 100)
print("True Positives:", TP)
print("False Positives:", FP)
print("False Negatives:", FN)
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")

Samples tested: 100
True Positives: 15
False Positives: 99
False Negatives: 32
Precision: 0.132
Recall: 0.319
F1-score: 0.186


In [21]:
# إخفاء أسماء الأشخاص من نص المكالمة
entities = name_model(redacted_transcript)

spans = [
    (item["start"], item["end"])
    for item in entities
    if item["entity_group"] == "PERSON"
]

for start, end in sorted(spans, reverse=True):
    redacted_transcript = (
        redacted_transcript[:start]
        + "[REDACTED]"
        + redacted_transcript[end:]
    )

print("النص النهائي بعد إخفاء الأسماء والأرقام:")
print(redacted_transcript)

النص النهائي بعد إخفاء الأسماء والأرقام:
 مرحبا بكم في خدمة العملاء كيف يمكنني مساعدتكم؟ مرحبا بكم، التصدف خصوص فاتورة هذا الشهر فقد خصم المبلغ مراتين من حساب المصرفين اعتدت عن هذا الازعاج، هل يمكنكم تزويد يبرقم الجوال المرطمز بالحساب؟ الرقم [REDACTED] وهذا الثالث مرة التصدف فيها بسبب المشكلة نفسها وفي كل مرة يقال لي أنها ستحال خلال اليومين ولم تحال ارى أن الطلب السابق ليزال قيد المراجعة قيد المراجعة من 3 اسابيه هذا غير مقبول اطلاقا ولم يعب لديه اي ثقى بيوعدكم اتفهم انزعاجكم تماما وسوى اتابع الطلب بنفسي وأوفيكم بالنتيجة خلال 48 ساعة لا اريد وعودة جديدة اريد استرداد مبلغ اليوم واتلب التحدث مع المشرف حالا حسنا سحاول المكالمة المشرف فورا ولا نغلق الخطق قبل ان يثلمها شكرا لك وان لم تحل المسأل اليوم فسأقدم شكرا رسمي


In [22]:
# اختبار إخفاء المعلومات الشخصية على عينة عشوائية
import random
import re

# اختيار عينة تحتوي على اسم أو رقم حساب أو معلومات شخصية
candidates = [
    sample for sample in dataset["test"]
    if any(
        entity["label"] in [
            "PERSON", "PHONE_NUMBER", "BANK_ACCOUNT_NUMBER"
        ]
        for entity in sample["entities"]
    )
]

sample = random.choice(candidates)
original_text = sample["text"]

# اكتشاف المعلومات الشخصية بالنموذجين
sitr_entities = ner(original_text)
name_entities = name_model(original_text)

spans = [
    (item["start"], item["end"])
    for item in sitr_entities + name_entities
]

# اكتشاف أرقام الجوال المكتوبة بأرقام متصلة أو بينها مسافات
for match in re.finditer(r"(?<!\d)0(?:\s*\d){8,11}(?!\d)", original_text):
    spans.append((match.start(), match.end()))

# دمج الأجزاء المتداخلة
spans.sort()
merged = []

for start, end in spans:
    if merged and start <= merged[-1][1]:
        merged[-1] = (merged[-1][0], max(merged[-1][1], end))
    else:
        merged.append((start, end))

# إخفاء المعلومات من نهاية النص إلى بدايته
redacted_text = original_text

for start, end in reversed(merged):
    redacted_text = (
        redacted_text[:start] + "[REDACTED]" + redacted_text[end:]
    )

print("النص قبل الإخفاء:")
print(original_text)

print("\nالنص بعد الإخفاء:")
print(redacted_text)

النص قبل الإخفاء:
يا زياد أبو غنام، الـ feature request هذي مو واضحة ممكن توضحلي أكثر؟

النص بعد الإخفاء:
يا [REDACTED]، الـ feature request هذي مو واضحة ممكن توضحلي أكثر؟


In [23]:
# اختبار إخفاء المعلومات الشخصية على 5 أمثلة عشوائية
import random
import re

random.seed(42)

candidates = [
    sample for sample in dataset["test"]
    if any(
        entity["label"] in [
            "PERSON", "PHONE_NUMBER", "BANK_ACCOUNT_NUMBER"
        ]
        for entity in sample["entities"]
    )
]

samples = random.sample(candidates, 5)

for i, sample in enumerate(samples, 1):
    original_text = sample["text"]

    # اكتشاف المعلومات الشخصية
    sitr_entities = ner(original_text)
    name_entities = name_model(original_text)

    spans = [
        (item["start"], item["end"])
        for item in sitr_entities + name_entities
    ]

    # اكتشاف أرقام الجوال المكتوبة بأرقام متصلة أو بينها مسافات
    for match in re.finditer(
        r"(?<!\d)0(?:\s*\d){8,11}(?!\d)", original_text
    ):
        spans.append((match.start(), match.end()))

    # دمج الأجزاء المتداخلة
    spans.sort()
    merged = []

    for start, end in spans:
        if merged and start <= merged[-1][1]:
            merged[-1] = (
                merged[-1][0],
                max(merged[-1][1], end)
            )
        else:
            merged.append((start, end))

    # إخفاء المعلومات
    redacted_text = original_text

    for start, end in reversed(merged):
        redacted_text = (
            redacted_text[:start]
            + "[REDACTED]"
            + redacted_text[end:]
        )

    print(f"\n{'=' * 50}")
    print(f"المثال {i}")
    print("قبل الإخفاء:")
    print(original_text)
    print("\nبعد الإخفاء:")
    print(redacted_text)


المثال 1
قبل الإخفاء:
الموظف: الأم اتصلت وقالت هتجيب شهادة الميلاد بكره...آسف، أقصد النهاردة، رقمها +٢٠ ١٥ ٤٦٩٦ ١٦٠٨

بعد الإخفاء:
الموظف: الأم اتصلت وقالت هتجيب شهادة الميلاد بكره...آسف، أقصد النهاردة، رقمها[REDACTED]

المثال 2
قبل الإخفاء:
Name: حمد آل كمال
Account ID: ٧٧٤٠٥٤٩٩٨
Phone: ٠٥٨ ٦٤٢ ١٢٨٦
التذكرة: أبي ألغي التذكرة حقتي لحفلة موسم الرياض، بالغلط حجزت تاريخ غير هذا اللي أبيه.

بعد الإخفاء:
Name: [REDACTED]
Account ID: [REDACTED]
Phone: [REDACTED]
التذكرة: أبي ألغي التذكرة حقتي لحفلة موسم [REDACTED]، بالغلط حجزت تاريخ غير هذا اللي أبيه.

المثال 3
قبل الإخفاء:
المندوب: صباح الخير أستاذ تيم أبو خطيب! بخصوص طلب التوصيل لعداد الكهرباء في ميدان البلدية ٥٩، حي الغبرة، مسقط، بناية ٦٨، شقة ٢٩، تم تقديم الطلب ومراجعته، لكن نحتاج منك تزويدنا بصورة واضحة عن الهوية الوطنية.

بعد الإخفاء:
[REDACTED]: صباح الخير أستاذ [REDACTED]! بخصوص طلب التوصيل لعداد الكهرباء في [REDACTED]، تم تقديم الطلب ومراجعته، لكن نحتاج منك تزويدنا بصورة واضحة عن الهوية الوطنية.

المثال 4
قبل الإخفاء:
العميل: ألو، 